<a href="https://colab.research.google.com/github/cynthiiaa/one-off-projects/blob/main/FineTuningWithKeras%26JAX.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip uninstall -y jax jaxlib ml-dtypes flax optax orbax-checkpoint
# !pip install --upgrade \
#   "jax[tpu]==0.5.3" \
#   "jaxlib==0.5.3" \
#   "ml-dtypes==0.5.0" \
#   "transformers<5.0.0" \ # no more JAX support for transformers 5+
#   datasets \
#   flax==0.10.0 \
#   optax==0.2.8 \
#   -i https://us-python.pkg.dev/ml-oss-artifacts-published/jax/simple/ \
#   -f https://storage.googleapis.com/jax-releases/libtpu_releases.html

I'm using Kera3 since it supports a JAX backend. Unfortunately, `transformers` 5+ no longer supports JAX or TensorFlow for the backend. But the commented out code above should work if you don't mind using a downgraded version of transformers.

In [ ]:
!pip install -q --upgrade keras keras-hub jax[tpu] -f https://storage.googleapis.com/jax-releases/libtpu_releases.html

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax"

import keras
import keras_hub
import jax
import jax.numpy as jnp
import numpy as np
from datasets import load_dataset

print("JAX Devices:", jax.devices())

/usr/local/lib/python3.13/dist-packages/jax/_src/cloud_tpu_init.py:90: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


JAX Devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]


# Fine Tuning with Keras3 and JAX

Look, fine-tuning is the same thing as training from scratch. The loop, loss function, optimizer, and backpropagation are identical. What makes it *fine-tuning* comes down to whether or not you used a pretrained model.

In this exercise, we call `from_preset("gpt2_base_en")` which downloads a model that OpenAI spent millions of dollars calculating. The model already knows English really well.

Instead of you feeding an LLM with petabytes of the entire internet, you can fine-tune a prebuilt model on your niche, highly specific dataset so it learns a specific behavior or format.

In [ ]:
model = keras_hub.models.GPT2CausalLM.from_preset("gpt2_base_en")
preprocessor = keras_hub.models.GPT2CausalLMPreprocessor.from_preset("gpt2_base_en", sequence_length=64)

In [ ]:
# we're using English quotes
data = load_dataset("Abirate/english_quotes", split="train[:2000]")
formatted_text = [f'"{q}" - {a}' for q, a in zip(data["quote"], data["author"])]
processed_data = preprocessor(formatted_text)

x_data = np.array(processed_data[0]["token_ids"])
mask_data = np.array(processed_data[0]["padding_mask"])
y_data = np.array(processed_data[1]) # preprocessor automatically shifts the targets


### Stateless Optimizer
Basically, a pure mathematical function that calculates weight updates without modifying any hidden internal variables in-place.

With PyTorch, optimizers like `Adam` keep a history of past gradients to smooth out the training. When you call `optimizer.step()`, the optimizer **mutates** the model's weights and overwrites its own internal trackers in-place. In functional programming this is a no-no.

The XLA compiler requires pure functions with **zero side-effects**.

To comply with JAX's rules, Keras 3 exposes a stateless API where the optimizer acts purely as an input-output function. It doesn't own nor store the trackers (the engineer does).

Momentum and velocity (sometimes called variance or second momentum) are techniques used by optimizes like `Adam` to solve gradient descent problems.

First momentum tracks the direction of the slope while second momentum tracks the steepness.

Velocity Example: if the parameter is on a massive, steep incline (high velocity), the optimizer automatically shrinks its learning rate.

A mask is a binary array of 1s and 0s that tells the neural network which tokens are real data and which are filler.

Loss is only calculated on the real tokens because padding is a formatting requirement, not part of the language.

The data that's fed into these models need to be strictly rectangular matrices. For example, if the context window is 64 tokens, then every single sequence fed into the accelerator (GPU/TPU) must be exactly 64 tokens long.

But real sentences vary in length. So a quote that has 3 tokens would need to have the remaining 61 slots filled with dummy tokens (aka padding). If not, the math will crash.

Since the loss function calculates how wrong the model was by comparing its predictions to the target `y` sequence, we need to pass the `mask`. Otherwise, the loss function will treat the padding zeros like legit vocabulary words the model was supposed to predict.

## Training

In [ ]:
keras_trainable = model.trainable_variables
keras_non_trainable = model.non_trainable_variables

trainable_vars = [jnp.array(v) for v in keras_trainable]
non_trainable_vars = [jnp.array(v) for v in keras_non_trainable]

# we'll use a stateless optimizer
optimizer = keras.optimizers.AdamW(learning_rate=5e-5) # create the optimizer skeleton
optimizer.build(keras_trainable) # initialize empty arrays to track momentum and velocity for every parameter
optimizer_vars = [jnp.array(v) for v in optimizer.variables] # the engineer now "owns" the state

loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

@jax.jit
def train_step(trainable, non_trainable, optional_vars, x, y, mask):
  def compute_loss(trainable_weights):
    logits, new_non_trainable = model.stateless_call(trainable_weights,
                                                     non_trainable,
                                                     {"token_ids": x, "padding_mask": mask})
    loss = loss_fn(y, logits, sample_weight=mask)
    return loss, new_non_trainable

  (loss, new_non_trainable), grads = jax.value_and_grad(compute_loss, has_aux=True)(trainable)
  new_trainable, new_optional_vars = optimizer.stateless_apply(optional_vars, grads, trainable)

  return new_trainable, new_non_trainable, new_optional_vars, loss

In [ ]:
batch_size = 16
num_samples = len(x_data)
steps_per_epoch = num_samples // batch_size

for epoch in range(3):
  perm = np.random.permutation(num_samples)

  for step in range(steps_per_epoch):
    batch_idx = perm[step * batch_size : (step + 1) * batch_size]

    bx = jnp.array(x_data[batch_idx])
    by = jnp.array(y_data[batch_idx])
    bmask = jnp.array(mask_data[batch_idx])

    trainable_vars, non_trainable_vars, optimizer_vars, loss = train_step(trainable_vars, non_trainable_vars, optimizer_vars, bx, by, bmask)

    if step % 50 == 0:
      print(f"Epoch {epoch+1} | Step {step} | Loss: {loss:.4f}")

Epoch 1 | Step 0 | Loss: 2.6157
Epoch 1 | Step 50 | Loss: 1.5422
Epoch 1 | Step 100 | Loss: 2.2908
Epoch 2 | Step 0 | Loss: 1.7014
Epoch 2 | Step 50 | Loss: 1.6948
Epoch 2 | Step 100 | Loss: 1.6995
Epoch 3 | Step 0 | Loss: 1.1707
Epoch 3 | Step 50 | Loss: 0.8526
Epoch 3 | Step 100 | Loss: 1.4447


## Inferencing

**Temperature** is the dial that controls how creative or predictable a language model's output is.

We scale the logits before they're converted to probablities 👇🏾.

When a model predicts the next word, it assigns a raw score to every word in the dictionary. Before those raw scores (logits) are converted into probablities (via an operation called Softmax), the model divides those scores by the temperature. $\frac{logits}{temperature}$

| Temperature | Effect | Best Used For |
| -------- | -------- | -------- |
| Low (<1.0) | Dividing by a decimal magnifies the differences. The distrubution spikes.<br>The model becomes highly predictable, repetitive, and favors the top choice. | coding, factual Q&A, math, formatting data|
| Default (1.0) | This changes nothing. | general conversation |
| High (>1.0)| Dividing by a whole number shrinks the differences. The distribution flattens.<br>The model takes risks and chooses "interesting" words. You increase creativity at the risk of getting back garbage.| poetry, brainstorming, creative writing |

In [ ]:
# overwrite trainable weights
for keras_var, jax_array in zip(model.trainable_variables, trainable_vars):
  keras_var.assign(jax_array)

# overwrite non-trainable weights
for keras_var, jax_array in zip(model.non_trainable_variables, non_trainable_vars):
  keras_var.assign(jax_array)

In [ ]:
sampler = keras_hub.samplers.TopKSampler(k=5, temperature=0.8)
model.compile(sampler=sampler)

In [ ]:
prompts = ['"The secret to learning', '"A successful', '"With great power']
results = model.generate(prompts, max_length=64)

i = 1
for r in results:
  print(f"{i}: {r}\n")
  i = i+1

1: "The secret to learning is not to read, but to learn.”" - Roy T. Bennett,

2: "A successful person is one who is willing to take risks, to be brave, to be kind. That's what makes a person.”" - Roy T. Bennett,

3: "With great power comes great responsibility. With great responsibility is responsibility.”" - Albert Einstein



Albert Einstein never said that famous quote 😭

# Steps on Fine-Tuning a Model

## 1) Your Setup
* Don't forget to set `KERAS_BACKEND` to your preferred backend.
* Load pre-trained weights by instantiating a base model from a model registry.
* Load the matching tokenizer/preprocessor for the base model to ensure your new data is tokenized using the same vocabulary mapping the model was originally trained on.

## 2) Data Formatting
* Map your raw dataset into the exact string format you want the model to learn.
* Pass the structured strings through the preprocessor. This automatically:
  * tokenizes the next text
  * pads sequences to a uniform length
  * generates a binary attention mask


## 3) Your Optimizer
* Pull the `trainable_variables` and `non_trainable_variables` out of the model.
* Build the optimizer on the framework variables and extract its internal tracking variables (`optimizer.variables`).
* Convert all extracted variables into pure JAX arrays.
  * This strips away the stateful, object-oriented shell and leaves the immutable inputs that the XLA compiler needs.


## 4) Training the Model
* Use Kera3's functional API (`model.stateless_call`) to pass the pure weight arrays and the data arrays.
  * This receives predictions without mutating the model.
* Apply the attention mask to your loss function.
* Use the stateless optimizer to calculate the parameter updates.

## 5) Deployment
* Once the training loop finishes, iterate through the model's original variables and use `.assign()` to overwrite them with the newly fine-tuned JAX arrays.
* Compile the updated model with a sampler like Top-K and Temperature.
* Either save the fine-tuned model or run it.